# Airtel Telecom Analytics — Phase 3: Exploratory Data Analysis

**Tools:** pandas · matplotlib · seaborn  
**Input:** `airtel_clean_data/*.csv`  
**Output:** `airtel_eda_outputs/` — charts and summary tables

---
**Analysis Sections:**
1. Customer base overview — demographics, plan mix, regional distribution
2. Churn analysis — churn rate by segment, reason breakdown, retention effectiveness
3. Billing patterns — revenue by plan, payment behavior, arrears concentration
4. Support operations — ticket volume, CSAT trends, resolution performance
5. Correlation heatmap — feature relationships for downstream modeling

**Business question:** Where does churn concentrate, what signals it, and what does it cost?

**Next:** `04_forecasting.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from config import CLEAN_DIR, EDA_DIR, CHART_DPI, CHART_STYLE, CHART_PALETTE, SEED
from utils import log, section, save_fig, snapshot

sns.set_theme(style=CHART_STYLE, palette=CHART_PALETTE)
np.random.seed(SEED)

section('AIRTEL TELECOM — Phase 3: Exploratory Data Analysis')

In [ ]:
# ─── LOAD CLEAN DATA ──────────────────────────────────────────────────────────

section('Loading Clean Data')

customers = pd.read_csv(f'{CLEAN_DIR}/customers_clean.csv', parse_dates=['joining_date'])
billing   = pd.read_csv(f'{CLEAN_DIR}/billing_clean.csv',   parse_dates=['payment_date'])
churn     = pd.read_csv(f'{CLEAN_DIR}/churn_events_clean.csv', parse_dates=['churn_date', 'reactivation_date'])
tickets   = pd.read_csv(f'{CLEAN_DIR}/support_tickets_clean.csv', parse_dates=['created_date', 'resolution_date'])

for name, df in [('customers', customers), ('billing', billing), ('churn', churn), ('tickets', tickets)]:
    snapshot(df, name)

log('\n  ✅ All clean datasets loaded')

---
## 1. Customer Base Overview

In [ ]:
# ─── PLAN MIX & REGIONAL DISTRIBUTION ────────────────────────────────────────

section('Customer Base Overview')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Customer Base Overview', fontsize=14, fontweight='bold', y=1.02)

# Plan type split
plan_counts = customers['plan_type'].value_counts()
axes[0].pie(
    plan_counts.values,
    labels=plan_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette(CHART_PALETTE, len(plan_counts))
)
axes[0].set_title('Plan Type Split')

# Regional distribution
region_counts = customers['region'].value_counts()
sns.barplot(x=region_counts.index, y=region_counts.values, ax=axes[1], palette=CHART_PALETTE)
axes[1].set_title('Customers by Region')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Count')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)

# Age distribution
axes[2].hist(customers['age'].dropna(), bins=20, color=sns.color_palette(CHART_PALETTE)[0], edgecolor='white')
axes[2].set_title('Age Distribution')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Count')
axes[2].axvline(customers['age'].median(), color='red', linestyle='--', linewidth=1.5, label=f"Median: {customers['age'].median():.0f}")
axes[2].legend(fontsize=9)

plt.tight_layout()
save_fig(EDA_DIR, '01_customer_overview', CHART_DPI)

log(f"  Plan mix: {plan_counts.to_dict()}")
log(f"  Median age: {customers['age'].median():.0f}")
log(f"  Active customers: {customers['is_active'].sum():,} / {len(customers):,} ({customers['is_active'].mean()*100:.1f}%)")

In [ ]:
# ─── ACQUISITION CHANNEL & CONTRACT TYPE ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Acquisition & Contract Mix', fontsize=14, fontweight='bold')

acq = customers['acquisition_channel'].value_counts()
sns.barplot(x=acq.values, y=acq.index, ax=axes[0], palette=CHART_PALETTE, orient='h')
axes[0].set_title('Acquisition Channel')
axes[0].set_xlabel('Customers')

contract = customers['contract_type'].value_counts()
axes[1].pie(
    contract.values,
    labels=contract.index,
    autopct='%1.1f%%',
    colors=sns.color_palette(CHART_PALETTE, len(contract))
)
axes[1].set_title('Contract Type')

plt.tight_layout()
save_fig(EDA_DIR, '02_acquisition_contract', CHART_DPI)

---
## 2. Churn Analysis

In [ ]:
# ─── CHURN RATE BY SEGMENT ────────────────────────────────────────────────────

section('Churn Analysis')

churned_ids = set(churn['customer_id'].unique())
customers['is_churned'] = customers['customer_id'].isin(churned_ids).astype(int)

overall_churn_rate = customers['is_churned'].mean() * 100
log(f'  Overall churn rate: {overall_churn_rate:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Churn Rate by Segment', fontsize=14, fontweight='bold')

# By plan type
churn_by_plan = customers.groupby('plan_type')['is_churned'].mean() * 100
sns.barplot(x=churn_by_plan.index, y=churn_by_plan.values, ax=axes[0], palette=CHART_PALETTE)
axes[0].set_title('Churn Rate by Plan Type')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].axhline(overall_churn_rate, color='red', linestyle='--', linewidth=1, label=f'Overall: {overall_churn_rate:.1f}%')
axes[0].legend(fontsize=8)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=9)

# By region
churn_by_region = customers.groupby('region')['is_churned'].mean() * 100
sns.barplot(x=churn_by_region.index, y=churn_by_region.values, ax=axes[1], palette=CHART_PALETTE)
axes[1].set_title('Churn Rate by Region')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].axhline(overall_churn_rate, color='red', linestyle='--', linewidth=1)
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=9)

# By contract type
churn_by_contract = customers.groupby('contract_type')['is_churned'].mean() * 100
sns.barplot(x=churn_by_contract.index, y=churn_by_contract.values, ax=axes[2], palette=CHART_PALETTE)
axes[2].set_title('Churn Rate by Contract Type')
axes[2].set_ylabel('Churn Rate (%)')
axes[2].axhline(overall_churn_rate, color='red', linestyle='--', linewidth=1)
for bar in axes[2].patches:
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=9)

plt.tight_layout()
save_fig(EDA_DIR, '03_churn_by_segment', CHART_DPI)

In [ ]:
# ─── CHURN REASONS & RETENTION ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Churn Reasons & Retention Outcomes', fontsize=14, fontweight='bold')

# Churn reasons
reasons = churn['churn_reason'].value_counts()
sns.barplot(x=reasons.values, y=reasons.index, ax=axes[0], palette=CHART_PALETTE, orient='h')
axes[0].set_title('Churn Reasons')
axes[0].set_xlabel('Count')

# Voluntary vs Involuntary
churn_type = churn['churn_type'].value_counts()
axes[1].pie(
    churn_type.values,
    labels=churn_type.index,
    autopct='%1.1f%%',
    colors=sns.color_palette(CHART_PALETTE, len(churn_type))
)
axes[1].set_title('Voluntary vs Involuntary Churn')

# Retention success
retention = churn['was_retained'].value_counts()
axes[2].pie(
    retention.values,
    labels=[f"{l} ({v})" for l, v in zip(retention.index, retention.values)],
    autopct='%1.1f%%',
    colors=sns.color_palette(CHART_PALETTE, len(retention))
)
axes[2].set_title('Retention Success Rate')

plt.tight_layout()
save_fig(EDA_DIR, '04_churn_reasons_retention', CHART_DPI)

retention_rate = (churn['was_retained'] == 'Yes').mean() * 100
log(f'  Retention success rate: {retention_rate:.1f}%')
log(f'  Top churn reason: {reasons.index[0]} ({reasons.iloc[0]} customers)')

In [ ]:
# ─── CHURN OVER TIME ──────────────────────────────────────────────────────────

churn['churn_month'] = churn['churn_date'].dt.to_period('M')
monthly_churn = churn.groupby('churn_month').size().reset_index(name='churn_count')
monthly_churn['churn_month_str'] = monthly_churn['churn_month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(len(monthly_churn)), monthly_churn['churn_count'],
                alpha=0.3, color=sns.color_palette(CHART_PALETTE)[0])
ax.plot(range(len(monthly_churn)), monthly_churn['churn_count'],
        marker='o', markersize=5, color=sns.color_palette(CHART_PALETTE)[0], linewidth=2)
ax.set_xticks(range(len(monthly_churn)))
ax.set_xticklabels(monthly_churn['churn_month_str'], rotation=45, ha='right', fontsize=8)
ax.set_title('Monthly Churn Events', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Count')
ax.set_xlabel('Month')
ax.axhline(monthly_churn['churn_count'].mean(), color='red', linestyle='--',
           linewidth=1.2, label=f"Avg: {monthly_churn['churn_count'].mean():.0f}/mo")
ax.legend()
plt.tight_layout()
save_fig(EDA_DIR, '05_monthly_churn_trend', CHART_DPI)

---
## 3. Billing Patterns

In [ ]:
# ─── BILLING OVERVIEW ─────────────────────────────────────────────────────────

section('Billing Analysis')

total_revenue   = billing['total_amount'].sum()
total_arrears   = billing['arrears'].sum()
unpaid_rate     = (billing['payment_status'] == 'Unpaid').mean() * 100

log(f'  Total revenue in dataset: ₹{total_revenue:,.0f}')
log(f'  Total arrears: ₹{total_arrears:,.0f}')
log(f'  Unpaid bill rate: {unpaid_rate:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Billing Patterns', fontsize=14, fontweight='bold')

# Payment status distribution
pay_status = billing['payment_status'].value_counts()
axes[0].pie(
    pay_status.values,
    labels=pay_status.index,
    autopct='%1.1f%%',
    colors=sns.color_palette(CHART_PALETTE, len(pay_status))
)
axes[0].set_title('Payment Status')

# Total amount distribution
axes[1].hist(billing['total_amount'].dropna(), bins=30,
             color=sns.color_palette(CHART_PALETTE)[1], edgecolor='white')
axes[1].set_title('Bill Amount Distribution')
axes[1].set_xlabel('Amount (₹)')
axes[1].set_ylabel('Count')
axes[1].axvline(billing['total_amount'].median(), color='red', linestyle='--',
                label=f"Median: ₹{billing['total_amount'].median():.0f}")
axes[1].legend(fontsize=9)

# Payment method
pay_method = billing['payment_method'].value_counts()
sns.barplot(x=pay_method.values, y=pay_method.index, ax=axes[2],
            palette=CHART_PALETTE, orient='h')
axes[2].set_title('Payment Method')
axes[2].set_xlabel('Count')

plt.tight_layout()
save_fig(EDA_DIR, '06_billing_overview', CHART_DPI)

In [ ]:
# ─── REVENUE BY PLAN & ARREARS ANALYSIS ───────────────────────────────────────

# Join billing with customer plan data
billing_with_plan = billing.merge(
    customers[['customer_id', 'plan_type', 'plan_name', 'region']],
    on='customer_id', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Revenue & Arrears by Plan', fontsize=14, fontweight='bold')

# Revenue by plan type
rev_by_plan = billing_with_plan.groupby('plan_type')['total_amount'].sum().sort_values(ascending=False)
sns.barplot(x=rev_by_plan.index, y=rev_by_plan.values, ax=axes[0], palette=CHART_PALETTE)
axes[0].set_title('Total Revenue by Plan Type')
axes[0].set_ylabel('Revenue (₹)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'₹{bar.get_height()/1000:.0f}K', ha='center', fontsize=9)

# Arrears by plan type
arrears_by_plan = billing_with_plan.groupby('plan_type')['arrears'].sum().sort_values(ascending=False)
sns.barplot(x=arrears_by_plan.index, y=arrears_by_plan.values, ax=axes[1],
            palette=['#e74c3c', '#e67e22'])
axes[1].set_title('Total Arrears by Plan Type')
axes[1].set_ylabel('Arrears (₹)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

plt.tight_layout()
save_fig(EDA_DIR, '07_revenue_arrears_by_plan', CHART_DPI)

---
## 4. Support Operations

In [ ]:
# ─── TICKET VOLUME & CSAT ─────────────────────────────────────────────────────

section('Support Operations Analysis')

avg_csat        = tickets['csat_score'].replace(0, np.nan).mean()
escalation_rate = tickets['escalated'].mean() * 100
fcr_rate        = tickets['first_contact_resolved'].mean() * 100

log(f'  Average CSAT: {avg_csat:.2f} / 5')
log(f'  Escalation rate: {escalation_rate:.1f}%')
log(f'  First contact resolution rate: {fcr_rate:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Support Operations Overview', fontsize=14, fontweight='bold')

# Tickets by issue category
issue_counts = tickets['issue_category'].value_counts()
sns.barplot(x=issue_counts.values, y=issue_counts.index, ax=axes[0],
            palette=CHART_PALETTE, orient='h')
axes[0].set_title('Tickets by Issue Category')
axes[0].set_xlabel('Count')

# CSAT distribution
csat_vals = tickets['csat_score'].replace(0, np.nan).dropna()
csat_counts = csat_vals.value_counts().sort_index()
sns.barplot(x=csat_counts.index.astype(str), y=csat_counts.values,
            ax=axes[1], palette=CHART_PALETTE)
axes[1].set_title('CSAT Score Distribution')
axes[1].set_xlabel('CSAT Score (1–5)')
axes[1].set_ylabel('Count')
axes[1].axvline(avg_csat - 1, color='red', linestyle='--', linewidth=1.5,
                label=f'Avg: {avg_csat:.2f}')
axes[1].legend(fontsize=9)

# Priority breakdown
priority = tickets['priority'].value_counts()
axes[2].pie(
    priority.values,
    labels=priority.index,
    autopct='%1.1f%%',
    colors=sns.color_palette(CHART_PALETTE, len(priority))
)
axes[2].set_title('Ticket Priority Mix')

plt.tight_layout()
save_fig(EDA_DIR, '08_support_overview', CHART_DPI)

In [ ]:
# ─── RESOLUTION PERFORMANCE ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Resolution Performance', fontsize=14, fontweight='bold')

# Days to resolve distribution
resolve_times = tickets['days_to_resolve'].dropna()
axes[0].hist(resolve_times, bins=20,
             color=sns.color_palette(CHART_PALETTE)[2], edgecolor='white')
axes[0].set_title('Days to Resolve Distribution')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Count')
axes[0].axvline(resolve_times.median(), color='red', linestyle='--',
                label=f'Median: {resolve_times.median():.1f} days')
axes[0].legend(fontsize=9)

# Avg handle time by channel
handle_by_channel = tickets.groupby('channel')['handle_time_mins'].mean().sort_values(ascending=False)
sns.barplot(x=handle_by_channel.values, y=handle_by_channel.index,
            ax=axes[1], palette=CHART_PALETTE, orient='h')
axes[1].set_title('Avg Handle Time by Channel (mins)')
axes[1].set_xlabel('Minutes')

plt.tight_layout()
save_fig(EDA_DIR, '09_resolution_performance', CHART_DPI)

---
## 5. Correlation Heatmap — Feature Relationships for Modeling

In [ ]:
# ─── BUILD CUSTOMER-LEVEL FEATURE TABLE ───────────────────────────────────────

section('Correlation Analysis')

# Billing aggregates per customer
billing_agg = billing.groupby('customer_id').agg(
    total_billed   = ('total_amount',  'sum'),
    avg_bill       = ('total_amount',  'mean'),
    unpaid_count   = ('payment_status', lambda x: (x == 'Unpaid').sum()),
    total_arrears  = ('arrears',        'sum'),
    total_discount = ('discount_applied', 'sum'),
).reset_index()

# Ticket aggregates per customer
ticket_agg = tickets.groupby('customer_id').agg(
    ticket_count    = ('ticket_id',    'count'),
    escalated_count = ('escalated',    'sum'),
    avg_csat        = ('csat_score',   lambda x: x.replace(0, np.nan).mean()),
    avg_handle_time = ('handle_time_mins', 'mean'),
).reset_index()

# Merge everything
features = customers[['customer_id', 'age', 'tenure_months', 'plan_price',
                       'is_active', 'is_churned']].copy()
features = features.merge(billing_agg, on='customer_id', how='left')
features = features.merge(ticket_agg,  on='customer_id', how='left')
features = features.fillna(0)

log(f'  Feature table: {features.shape[0]:,} rows × {features.shape[1]} columns')
snapshot(features, 'feature_table')

# Export for Phase 4
features.to_csv(f'{CLEAN_DIR}/customer_features.csv', index=False)
log(f'  ✅ Saved → {CLEAN_DIR}/customer_features.csv')

In [ ]:
# ─── CORRELATION HEATMAP ──────────────────────────────────────────────────────

numeric_cols = ['age', 'tenure_months', 'plan_price', 'is_churned',
                'total_billed', 'avg_bill', 'unpaid_count', 'total_arrears',
                'ticket_count', 'escalated_count', 'avg_csat']

corr_matrix = features[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle mask
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Heatmap\n(relationship strength with is_churned and other signals)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(EDA_DIR, '10_correlation_heatmap', CHART_DPI)

# Log top correlates with churn
churn_corr = corr_matrix['is_churned'].drop('is_churned').sort_values(key=abs, ascending=False)
log('\n  Top correlates with churn:')
for feat, val in churn_corr.head(5).items():
    log(f'    {feat:<25} {val:+.3f}')

In [ ]:
# ─── CHURN VS KEY SIGNALS — BOXPLOTS ──────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Key Signal Distribution: Churned vs Retained', fontsize=14, fontweight='bold')

signals = ['unpaid_count', 'total_arrears', 'ticket_count', 'tenure_months']
labels  = ['Unpaid Bills', 'Total Arrears (₹)', 'Support Tickets', 'Tenure (months)']

for ax, col, label in zip(axes, signals, labels):
    churned_vals  = features.loc[features['is_churned'] == 1, col]
    retained_vals = features.loc[features['is_churned'] == 0, col]
    ax.boxplot(
        [retained_vals, churned_vals],
        labels=['Retained', 'Churned'],
        patch_artist=True,
        boxprops=dict(facecolor=sns.color_palette(CHART_PALETTE)[0], alpha=0.6),
        medianprops=dict(color='red', linewidth=2)
    )
    ax.set_title(label, fontsize=10)
    ax.set_ylabel(label)

plt.tight_layout()
save_fig(EDA_DIR, '11_churn_signal_boxplots', CHART_DPI)

In [ ]:
# ─── EDA SUMMARY ──────────────────────────────────────────────────────────────

from utils import save_report

section('Phase 3 Complete')
log(f'''
  Charts saved → {EDA_DIR}/
    01_customer_overview.png
    02_acquisition_contract.png
    03_churn_by_segment.png
    04_churn_reasons_retention.png
    05_monthly_churn_trend.png
    06_billing_overview.png
    07_revenue_arrears_by_plan.png
    08_support_overview.png
    09_resolution_performance.png
    10_correlation_heatmap.png
    11_churn_signal_boxplots.png

  Feature table exported → {CLEAN_DIR}/customer_features.csv
  (Used as input for Phase 4 PyCaret modeling)

  NEXT → Run 04_forecasting.ipynb
''')

save_report(f'{EDA_DIR}/phase3_report.txt')